# ARTI 308 – Lab 5: Feature Engineering (Classification)
## Order Status Prediction using a Talabat-style Orders Dataset

### Lab focus
This dataset is already clean (no missing values, no duplicate rows, consistent data types).  
In this lab, we focus on **feature engineering** for a classification task, not on data cleaning.

### Objective
Build a baseline model to predict `Order_Status` (Delivered, Cancelled, In Transit) and learn how feature engineering choices affect model performance and feature importance.

In this lab we will:
1) Load and inspect the dataset  
2) Define the target and select usable predictors (avoid leakage features)  
3) Engineer new features (time-based, price-based, distance-based)  
4) Encode categorical features  
5) Train a baseline **Random Forest** classifier  
6) Interpret performance and feature importance  

**Prepared by:** Mohammed Hussain Alkahmis  
**Student ID:** 2240005057

## 1. Setup and imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the dataset

In [ ]:
DATA_PATH = "talabat_enhanced_orders.csv"  # ensure the file is in the same folder as this notebook
df = pd.read_csv(DATA_PATH)

df.head(10)


The first rows confirm that the dataset loaded correctly.  
Each row represents one food delivery order, including information about the customer, restaurant, driver, and order outcome (`Order_Status`).

## 3. Quick dataset checks (cleanliness confirmation)

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count").T)

print("\nDuplicate rows:", df.duplicated().sum())

We confirm the dataset is clean: no missing values and no duplicated rows.  
Therefore, we will spend our effort on feature engineering rather than cleaning.

## 4. Target variable and class balance

In [ ]:
target_col = "Order_Status"
df[target_col].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Count")
plt.show()

This bar chart shows whether the classes are balanced.  
If one class dominates, the model may learn to predict that class more often, so we must interpret accuracy carefully and also look at the confusion matrix.

## 5. Identify feature types

In [ ]:
df.dtypes

We have a mixture of numerical features (e.g., `Quantity`, `Total_Price`, distances) and categorical features (e.g., `City`, `Payment_Method`, `Traffic_Level`).  
This is a common real-world situation where feature engineering and encoding become essential.


## 6. Leakage awareness (important)

When designing a prediction task, we must avoid using features that would not be available at prediction time.

For example, if we want to predict the order status **right after the customer places the order**, we should not use:
- `Delivery_Time` (known only later)
- `Delivery_Duration_Minutes` (known only after delivery)

In this lab, we will **exclude** obvious leakage features and focus on information that is typically available early in the order lifecycle.


## 7. Feature engineering

### 7.1 Time-based features from `Order_Time`
We convert `Order_Time` into a datetime, then extract:
- hour of day  
- day of week  
- weekend flag  
- peak hour flag (example rule: lunch and dinner periods)


In [ ]:
df_fe = df.copy()

# Parse time columns
df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"], errors="coerce")

df_fe["order_hour"] = df_fe["Order_Time"].dt.hour
df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek  # Monday=0, Sunday=6
df_fe["is_weekend"] = df_fe["order_dayofweek"].isin([5,6]).astype(int)

# Simple peak-hour rule (you can adjust based on local context):
# Lunch: 12-15, Dinner: 19-23
df_fe["is_peak_hour"] = df_fe["order_hour"].isin(list(range(12,16)) + list(range(19,24))).astype(int)

df_fe[["Order_Time","order_hour","order_dayofweek","is_weekend","is_peak_hour"]].head(10)


We transformed the original timestamp into multiple meaningful features.  
Models often learn better from these engineered features than from raw timestamps.


### 7.2 Price-based features
We create a feature that captures the price per item:
`price_per_item = Total_Price / Quantity`

This can help the model differentiate between an expensive order with few items and a cheaper order with many items.


In [ ]:
df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]
df_fe[["Quantity","Total_Price","price_per_item"]].head(10)


`price_per_item` is a derived feature that may reflect restaurant type, item category, or order complexity.  
It is an example of business-driven feature engineering.


### 7.3 Optional: Haversine distance from GPS coordinates
The dataset already includes `Delivery_Distance_km`.  
However, if latitude/longitude columns exist, we can also compute an additional distance feature using the Haversine formula.

This section is **optional** and will only run if the coordinate columns exist.


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometers."""
    R = 6371.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

coord_cols = {"Restaurant_Lat","Restaurant_Lon","Customer_Lat","Customer_Lon"}
if coord_cols.issubset(set(df_fe.columns)):
    df_fe["haversine_rest_to_cust_km"] = haversine_km(
        df_fe["Restaurant_Lat"], df_fe["Restaurant_Lon"],
        df_fe["Customer_Lat"], df_fe["Customer_Lon"]
    )
    display(df_fe[["Delivery_Distance_km","haversine_rest_to_cust_km"]].head(10))
else:
    print("Coordinate columns not found. Skipping Haversine feature.")


If computed, `haversine_rest_to_cust_km` is a physics-based distance derived from coordinates.  
It can be used as an additional engineered feature, and it can also be compared with `Delivery_Distance_km` to understand how the dataset’s provided distance was generated.


### 7.4 Reducing high-cardinality categories (example: `Item_Name`)
`Item_Name` may have many unique values. If we one-hot encode all items, the feature space becomes huge.

A common feature engineering approach is to keep the most frequent categories and map the rest to `Other`.


In [ ]:
if "Item_Name" in df_fe.columns:
    top_k = 20
    top_items = df_fe["Item_Name"].value_counts().head(top_k).index
    df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other")
    print("Unique Item_Name:", df_fe["Item_Name"].nunique())
    print("Unique Item_Name_reduced:", df_fe["Item_Name_reduced"].nunique())
    df_fe[["Item_Name","Item_Name_reduced"]].head(10)
else:
    print("Item_Name column not found.")


We reduced the cardinality of a text category feature.  
This often improves model stability and reduces overfitting, especially for baseline models.


## 8. Discretization (binning)

Discretization converts a continuous numerical feature into categories (bins).  
This can help some models capture non-linear relationships, and it also improves interpretability.

Here we discretize `Total_Price` into simple tiers.


In [ ]:
df_fe["price_tier"] = pd.cut(
    df_fe["Total_Price"],
    bins=[0, 100, 250, 500, np.inf],
    labels=["low","medium","high","very_high"]
)

df_fe[["Total_Price","price_tier"]].head(10)


`price_tier` groups numeric values into understandable categories.  
This may help capture patterns such as higher cancellation rates for very expensive orders, if such a trend exists.


## 9. Prepare features for modeling

We now select our predictors.

We will drop:
- IDs that do not represent meaningful signals by themselves
- Obvious leakage features (`Delivery_Time`, `Delivery_Duration_Minutes`)

We will keep:
- early-available numeric and categorical variables
- engineered features


In [ ]:
drop_cols = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes",
    "Item_Name"  # we replaced it with Item_Name_reduced
]

# keep only columns that exist (safe for future versions of the dataset)
drop_cols = [c for c in drop_cols if c in df_fe.columns]

X = df_fe.drop(columns=drop_cols + [target_col])
y = df_fe[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


We prepared a feature matrix `X` and a target vector `y`.  
The feature matrix includes engineered features such as time-based indicators, price per item, and reduced item category.


## 10. Split into train and test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


We use stratified splitting to keep class proportions similar in train and test sets.  
This makes evaluation more reliable for classification problems with imbalanced classes.


## 11. Encoding and baseline model (Random Forest)

### Why encoding?
Machine learning models require numerical input.  
Categorical variables must be converted into numbers. Here we use **One-Hot Encoding** for nominal categories.

### Why Random Forest for this lab?
We use Random Forest as a baseline because:
- it handles mixed features well
- it is robust for teaching purposes
- it provides feature importance to help us interpret engineered features


In [ ]:
# Identify categorical and numerical columns automatically
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

rf = RandomForestClassifier(
    n_estimators=10,
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

model

## 12. Train the model and evaluate

In [ ]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(baseline_accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


Accuracy gives a general sense of performance, but the classification report is more informative.  
Precision answers: when the model predicts a class, how often is it correct?  
Recall answers: out of all real cases of a class, how many did the model find?

The confusion matrix shows which classes the model confuses most often.


## 13. Feature importance (What mattered the most?)

Random Forest provides a built-in feature importance score.  
Because we used one-hot encoding, each categorical value becomes its own feature.  
We will extract the final feature names and plot the top importances.


In [ ]:
# Get feature names after preprocessing
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols) if len(categorical_cols) > 0 else np.array([])
all_feature_names = np.concatenate([cat_feature_names, np.array(numeric_cols)])

importances = model.named_steps["rf"].feature_importances_

fi = (pd.DataFrame({"feature": all_feature_names, "importance": importances})
        .sort_values("importance", ascending=False))

fi.head(15)


In [ ]:
plt.figure(figsize=(8,5))
top_n = 15
sns.barplot(data=fi.head(top_n), x="importance", y="feature")
plt.title(f"Top {top_n} Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


This chart helps us understand which engineered and original features contributed most to predicting `Order_Status`.  
A high importance score suggests the feature provides useful signal, but it does not automatically imply a causal relationship.


## 14. Optional: Feature selection using SelectFromModel

We can select a subset of features using model-based selection.  
This is optional and mainly used to illustrate the concept of feature selection after feature engineering.


In [ ]:
from sklearn.feature_selection import SelectFromModel

# Build a new pipeline that selects features based on RF importances
selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=10, random_state=42, n_jobs=1, class_weight="balanced_subsample"
    ),
    threshold="median"  # keep features above the median importance
)

model_fs = Pipeline(steps=[
    ("preprocess", preprocess),
    ("select", selector),
    ("rf", RandomForestClassifier(
        n_estimators=10, random_state=42, n_jobs=1, class_weight="balanced_subsample"
    ))
])

model_fs.fit(X_train, y_train)
y_pred_fs = model_fs.predict(X_test)
fs_accuracy = accuracy_score(y_test, y_pred_fs)

print("Accuracy (with feature selection):", round(fs_accuracy, 4))
print("\nClassification Report (with feature selection):")
print(classification_report(y_test, y_pred_fs))

If performance stays similar, feature selection may help simplify the model with minimal accuracy loss.  
If performance drops, it may indicate that important information was removed.

## 15. Student tasks

### Task 1
Create one new engineered feature that may help predict `Order_Status`, then justify it.

### Task 2
Try a different rule for `is_peak_hour` and compare the performance with the baseline.

### Task 3
Change `top_k` in `Item_Name_reduced` (for example 10, 30, 50) and compare:
- accuracy
- top feature importances

### Task 4
Run the optional feature selection section and explain whether it was beneficial in this case.

Below are the completed answers using the uploaded dataset.

### Task 1 Solution — New engineered feature: `driver_close_to_restaurant`

I created a new binary feature called `driver_close_to_restaurant`. It is equal to **1** when the driver's location is within **3 km** of the restaurant at order time, and **0** otherwise. This feature is useful because it captures operational readiness: if a driver is already close to the restaurant, pickup can happen faster and the order is more likely to be completed smoothly. Even though the dataset already contains delivery distance from restaurant to customer, it does not directly measure how far the driver is from the pickup point. So this engineered feature adds a different business signal related to dispatch efficiency.

In [ ]:
def build_model_and_score(df_model, target_col="Order_Status"):
    drop_cols_local = [
        "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
        "Order_Time", "Delivery_Time", "Delivery_Duration_Minutes",
        "Item_Name"
    ]

    feature_cols_local = [c for c in df_model.columns if c not in drop_cols_local + [target_col]]
    X_local = df_model[feature_cols_local].copy()
    y_local = df_model[target_col].copy()

    X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
        X_local, y_local,
        test_size=0.2,
        random_state=42,
        stratify=y_local
    )

    categorical_local = X_train_local.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_local = X_train_local.select_dtypes(include=[np.number, "bool"]).columns.tolist()

    preprocess_local = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_local),
            ("num", "passthrough", numeric_local),
        ]
    )

    model_local = Pipeline(steps=[
        ("preprocess", preprocess_local),
        ("rf", RandomForestClassifier(
            n_estimators=10,
            random_state=42,
            n_jobs=1,
            class_weight="balanced_subsample"
        ))
    ])

    model_local.fit(X_train_local, y_train_local)
    pred_local = model_local.predict(X_test_local)
    acc_local = accuracy_score(y_test_local, pred_local)

    ohe_local = model_local.named_steps["preprocess"].named_transformers_["cat"]
    cat_feature_names_local = (
        ohe_local.get_feature_names_out(categorical_local) if len(categorical_local) > 0 else np.array([])
    )
    all_feature_names_local = np.concatenate([cat_feature_names_local, np.array(numeric_local)])
    importances_local = model_local.named_steps["rf"].feature_importances_

    fi_local = (
        pd.DataFrame({"feature": all_feature_names_local, "importance": importances_local})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    return acc_local, fi_local

task1_df = df_fe.copy()
task1_df["driver_to_restaurant_km"] = haversine_km(
    task1_df["Driver_Lat"], task1_df["Driver_Lon"],
    task1_df["Restaurant_Lat"], task1_df["Restaurant_Lon"]
)
task1_df["driver_close_to_restaurant"] = (task1_df["driver_to_restaurant_km"] < 3).astype(int)

task1_accuracy, task1_fi = build_model_and_score(task1_df)

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Task 1 accuracy with driver_close_to_restaurant:", round(task1_accuracy, 4))
display(task1_fi.head(10))

**Task 1 discussion:**  
The new feature produced a very small improvement over the baseline in this experiment. That means the signal is weak, but still reasonable from a business perspective. In real delivery systems, the driver's proximity to the restaurant is operationally meaningful, so this remains a valid engineered feature even if the improvement is modest.

### Task 2 Solution — Different rule for `is_peak_hour`

For the baseline notebook, peak hours were defined as:
- lunch: **12, 13, 14**
- dinner: **19, 20, 21**

I tested a broader rule:
- lunch: **11, 12, 13, 14**
- dinner: **18, 19, 20, 21, 22**

This expanded definition may capture more realistic busy periods around meal times.

In [ ]:
task2_df = df.copy()

task2_df["Order_Time"] = pd.to_datetime(task2_df["Order_Time"], errors="coerce")
task2_df["order_hour"] = task2_df["Order_Time"].dt.hour
task2_df["order_dayofweek"] = task2_df["Order_Time"].dt.dayofweek
task2_df["is_weekend"] = task2_df["order_dayofweek"].isin([4, 5]).astype(int)

expanded_peak_hours = [11, 12, 13, 14, 18, 19, 20, 21, 22]
task2_df["is_peak_hour"] = task2_df["order_hour"].isin(expanded_peak_hours).astype(int)

task2_df["price_per_item"] = task2_df["Total_Price"] / task2_df["Quantity"]
task2_df["haversine_rest_to_cust_km"] = haversine_km(
    task2_df["Restaurant_Lat"], task2_df["Restaurant_Lon"],
    task2_df["Customer_Lat"], task2_df["Customer_Lon"]
)

top_items_20 = task2_df["Item_Name"].value_counts().head(20).index
task2_df["Item_Name_reduced"] = np.where(task2_df["Item_Name"].isin(top_items_20), task2_df["Item_Name"], "Other")

task2_df["price_tier"] = pd.cut(
    task2_df["Total_Price"],
    bins=[0, 100, 250, 500, np.inf],
    labels=["low", "medium", "high", "very_high"]
)

task2_accuracy, task2_fi = build_model_and_score(task2_df)

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Expanded peak-hour accuracy:", round(task2_accuracy, 4))
display(task2_fi.head(10))

**Task 2 discussion:**  
The expanded peak-hour rule gave a slightly higher accuracy than the baseline. This suggests that orders just before or just after the original lunch/dinner windows still behave like busy-period orders, so a wider definition of peak hours is more realistic for this dataset.

### Task 3 Solution — Compare `top_k` values in `Item_Name_reduced`

In [ ]:
top_k_results = []

for k in [10, 30, 50]:
    task3_df = df.copy()

    task3_df["Order_Time"] = pd.to_datetime(task3_df["Order_Time"], errors="coerce")
    task3_df["order_hour"] = task3_df["Order_Time"].dt.hour
    task3_df["order_dayofweek"] = task3_df["Order_Time"].dt.dayofweek
    task3_df["is_weekend"] = task3_df["order_dayofweek"].isin([4, 5]).astype(int)
    task3_df["is_peak_hour"] = task3_df["order_hour"].isin([12, 13, 14, 19, 20, 21]).astype(int)
    task3_df["price_per_item"] = task3_df["Total_Price"] / task3_df["Quantity"]
    task3_df["haversine_rest_to_cust_km"] = haversine_km(
        task3_df["Restaurant_Lat"], task3_df["Restaurant_Lon"],
        task3_df["Customer_Lat"], task3_df["Customer_Lon"]
    )

    top_items_k = task3_df["Item_Name"].value_counts().head(k).index
    task3_df["Item_Name_reduced"] = np.where(task3_df["Item_Name"].isin(top_items_k), task3_df["Item_Name"], "Other")

    task3_df["price_tier"] = pd.cut(
        task3_df["Total_Price"],
        bins=[0, 100, 250, 500, np.inf],
        labels=["low", "medium", "high", "very_high"]
    )

    acc_k, fi_k = build_model_and_score(task3_df)

    top_k_results.append({
        "top_k": k,
        "accuracy": round(acc_k, 4),
        "top_5_features": ", ".join(fi_k.head(5)["feature"].tolist())
    })

task3_results_df = pd.DataFrame(top_k_results)
display(task3_results_df)

**Task 3 discussion:**  
Changing `top_k` from 10 to 30 to 50 did not materially change the model accuracy in this experiment. The top feature importances also stayed almost the same, which indicates that `Item_Name_reduced` was not one of the strongest signals compared with price, location, and distance-related variables. So for this dataset, increasing `top_k` did not provide a clear benefit.

### Task 4 Solution — Feature selection

In [ ]:
print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Accuracy with feature selection:", round(fs_accuracy, 4))
print("Accuracy difference (feature selection - baseline):", round(fs_accuracy - baseline_accuracy, 4))

**Task 4 discussion:**  
Feature selection was **not beneficial** here. The feature-selection model achieved a slightly lower accuracy than the baseline model. That suggests the removed features still carried some useful information, or that the baseline feature space was already manageable enough for Random Forest. Therefore, in this case, keeping the full engineered feature set was the better choice.

## Wrap-up
In this lab, the dataset was already clean, so our focus was on feature engineering.  
We engineered time-based, price-based, and category-reduction features, then evaluated a baseline classifier and interpreted feature importance.
